#Ingest the Results csv file
1. Read the file using spark DataframeReader API
2. Add metadata columns 
   - Source File
   - Ingestion Timestamp
3. Write to Bronze Delta Tables


### Read the CSV using Spark Dataframe API


In [0]:
%run ../00_common/01_environment_config


In [0]:
%run ../00_common/02_bronze_helpers

In [0]:
table_name = f"{catalog_name}.{bronze_schema}.results"
source_file = f"{landing_folder_path}/results"

In [0]:
source_file

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,DateType,IntegerType,FloatType



results_schema = StructType([
    StructField("date",DateType()),
    StructField("raceName",StringType()),
    StructField("round", IntegerType()),
    StructField("season",IntegerType()),
    StructField("url",StringType()),
    StructField("constructorId",StringType()),
    StructField("driverId",StringType()),
    StructField("grid",IntegerType()),
    StructField("laps",IntegerType()),
    StructField("number",IntegerType()),
    StructField("points",FloatType()),
    StructField("position",IntegerType()),
    StructField("positionText",StringType()),
    StructField("status",StringType())
])


In [0]:
results_df=(
    spark.read
    .format("json")
    #.option("header","True")
    #.option("inferSchema", "True")
    .option("mode", "FAILFAST")
    .schema(results_schema)
    .load(source_file))


In [0]:
display(results_df)

### Adding 2 new colums
- Source File
- Ingestion Timestamp


In [0]:

results_final_df = new_columns_insertion(results_df)
display(results_final_df)

In [0]:
(results_final_df
.write
.format("delta")
.mode("overwrite")
.saveAsTable(table_name))

In [0]:
display(spark.table(table_name))

In [0]:
%sql

Select season, count(*) from formula1.bronze.results 
group by season
order by season

In [0]:
%sql
Select * from formula1.bronze.results where season is null;